In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
Page_index_api = os.getenv("Page_index_api")

In [3]:
from pageindex import PageIndexClient
from langchain.chat_models import init_chat_model

pi_client = PageIndexClient(api_key =Page_index_api )

model = init_chat_model("google_genai:gemini-2.5-flash-lite" , api_key = GOOGLE_API_KEY)

In [4]:
model.invoke("hello")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db0d2-73d3-7492-8b78-9911832af6ab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

# Upload the pdfs docs

In [ ]:
import os
from pageindex import PageIndexClient

data_folder = "../data"


for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(data_folder, file_name)
        
        print(f"Uploading: {file_name}")
        res = pi_client.submit_document(file_path)

10

In [8]:
res = pi_client.list_documents()

In [11]:
len(res['documents'])

3

In [13]:
res['documents'][0]

{'id': 'pi-cmnnmjex1016w01qv2ej4ay36',
 'name': 'projects_portfolio_in_github.pdf',
 'description': 'This document outlines a data science portfolio showcasing projects in data analysis, visualization, machine learning, recommendation systems, time series forecasting, deep learning, and natural language processing.',
 'status': 'completed',
 'createdAt': '2026-04-06T20:09:43.265000',
 'pageNum': 5,
 'folderId': None}

# submit_functions

In [14]:
import os
from loguru import logger

def submit_folder_to_pi(pi_client: PageIndexClient, folder_path: str):
    try:
        if not os.path.isdir(folder_path):
            raise ValueError(f"Invalid folder path: {folder_path}")

        responses = []

        for file_name in os.listdir(folder_path):
            if file_name.endswith(".pdf"):
                file_path = os.path.join(folder_path, file_name)

                res = pi_client.submit_document(file_path)

                if res:
                    responses.append(res)

        logger.info(f"Uploaded {len(responses)} files from folder.")

        return responses

    except Exception:
        logger.exception("Failed to upload folder")
        return []


In [16]:
def submit_file_to_pi(pi_client: PageIndexClient, file_path: str):
    try:
        if not os.path.isfile(file_path):
            raise ValueError(f"Invalid file path: {file_path}")
        if not file_path.endswith(".pdf"):
            raise ValueError("Only PDF files are supported.")
            
        logger.info(f"Uploading file: {file_path}")

        response = pi_client.submit_document(file_path)
        logger.success(f"Uploaded successfully: {file_path}")
        return response

    except Exception:
        logger.exception(f"Failed to upload file: {file_path}")
        return {}

# data ingestion class

In [17]:
import os
from typing import List, Optional
from dotenv import load_dotenv
from loguru import logger
from pageindex import PageIndexClient


class DataIngestion:

    def __init__(self, api_key: Optional[str] = None):
        load_dotenv()

        self.api_key = api_key or os.getenv("Page_index_api")

        if not self.api_key:
            raise ValueError("PageIndex API key is missing.")

        self.pi_client = PageIndexClient(api_key=self.api_key)


    # Upload Single File
    def upload_file(self, file_path: str) -> dict:
        submit_file_to_pi(self.pi_client , file_path )


    # Upload Folder
    def upload_folder(self, folder_path: str) -> List[dict]:
        submit_folder_to_pi(self.pi_client ,folder_path )

    